In [ ]:
!pip install unsloth "xformers"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 50.0 MB/s et

In [ ]:
from unsloth import FastLanguageModel
import transformers

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import numpy as np
import torch
import random
import transformers
import gc
import pandas as pd
import re
import json

In [ ]:
from huggingface_hub import login
### add your huggingface token here


In [ ]:
#from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "meta-llama/Meta-Llama-3-8B-instruct"
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
def create_text_generation_pipeline(model, tokenizer, temperature=1.0, max_new_tokens=1):
    """
    Creates a text-generation pipeline with the given model and tokenizer.

    Args:
        model: The preloaded model for text generation.
        tokenizer: The corresponding tokenizer.
        temperature (float): Sampling temperature for generation (default: 1.0).
        max_new_tokens (int): Maximum number of tokens to generate (default: 1024).

    Returns:
        A transformers pipeline object for text generation.
    """
    return transformers.pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        trust_remote_code=True,
        pad_token_id=0,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        do_sample=True
    )

# Example usage:
# pipe = create_text_generation_pipeline(model, tokenizer)


In [ ]:
model,tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=8192,
        dtype=None,
        load_in_4bit=True,
    )
FastLanguageModel.for_inference(model)
pipe = create_text_generation_pipeline(model, tokenizer)


==((====))==  Unsloth 2025.6.11: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
letter_token_ids = {
        "U": tokenizer("U", add_special_tokens=False)['input_ids'][0],
        "P": tokenizer("P", add_special_tokens=False)['input_ids'][0],
    }

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
test_data=pd.read_csv('/content/drive/MyDrive/centaur/reversal_learning/rw_reversal_learning.csv')

In [ ]:
test_data['choice'] = test_data['choice'].map({0: 'U', 1: 'P'})

In [ ]:
def generate_seeds(num_seeds=20, seed=42):
    """Generates a list of random seeds.

    Args:
        num_seeds: The number of seeds to generate.
        seed: The initial seed for the random number generator (for reproducibility).

    Returns:
        A list of random integer seeds.
    """
    random.seed(seed)  # Set initial seed for reproducibility
    seeds = [random.randint(1, 100000) for _ in range(num_seeds)]
    return seeds

In [ ]:
seeds=generate_seeds(num_seeds=32)

In [ ]:
def extract_model_choice(raw_response: str) -> str:
    """
    Extracts choice ('I' or 'H') from model's raw response text.
    Handles JSON and loose formats robustly.
    """
    try:
        # First try direct JSON parsing
        response_data = json.loads(raw_response)
        choice = response_data.get("choice", "").strip().upper()
        if choice in {"U", "P"}:
            return choice

    except json.JSONDecodeError:
        # Fallback: Search for JSON pattern in text
        json_match = re.search(r'{\s*"choice"\s*:\s*"?(U|P)"?\s*}', raw_response, re.IGNORECASE)
        if json_match:
            response_data = json.loads(json_match.group().replace("'", '"'))  # normalize quotes
            choice = response_data.get("choice", "").strip().upper()
            if choice in {"U", "P"}:
                return choice

    # Final fallback: Find first standalone I or H
    char_match = re.search(r'\b[UuPp]\b', raw_response)
    if char_match:
        return char_match.group().upper()

    raise ValueError("No valid choice ('U' or 'P') found in model response")

In [ ]:
def format_past_trials(past_trials: list) -> str:
    """Formats past trial data for the prompt by listing choice, reward, and cumulative reward."""
    return "".join(
        f"- In trial {trial['trial']} you chose {trial['choice']} and you got {trial['reward']}, Cumulative Reward: {trial['cumulative_reward']}\n"
        for trial in past_trials
    )


In [ ]:
def build_slot_prompt(current_trial: int, past_trials: list, total_trials: int) -> str:
    recent_trials = past_trials[-5:] if len(past_trials) > 5 else past_trials
    formatted_trials = format_past_trials(recent_trials)

    return f"""<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>

You are a participant in a two-armed bandit task. Your goal is to **maximize total rewards** over time.
The environment may change unpredictably, and past success does not guarantee future results.

<|eot_id|>

<|start_header_id|>user<|end_header_id|>

# Task Parameters
- Trial {current_trial} of {total_trials}
- Choose between: [U] or [P]
- Possible outcomes: 1 (success) or 0 (failure)
- Reward probabilities may change at any time

# Recent History
{formatted_trials}

# Required Response
Select your next choice to maximize rewards:

<|response_format|>
{{"choice": "
"""


In [ ]:
def generate(prompt, pipe, model, tokenizer, human_choice=None):
    """
    Generates model choice and computes log-likelihood of human choice.

    Args:
        prompt (str or list): The task prompt shown to the model.
        pipe (transformers.pipeline): Text generation pipeline.
        model (transformers.PreTrainedModel): The language model.
        tokenizer (transformers.PreTrainedTokenizer): The tokenizer for the model.
        human_choice (str): 'H' or 'I', the actual human response (optional).

    Returns:
        choice (str): The model's predicted choice (H or I).
        log_likelihood (float or None): Log-likelihood of the human choice.
    """
    import torch

    # Ensure prompt is a string
    prompt_str = "".join(prompt) if isinstance(prompt, list) else prompt

    # Generate model output
    outputs = pipe(prompt_str)
    full_text = outputs[0]['generated_text']

    # Extract model's choice (you should define this function to parse model's JSON)
    choice = extract_model_choice(full_text)  # e.g., returns "H" or "I"

    log_likelihood = None
    if human_choice and human_choice in ["U", "P"]:
        #print('here')
        # Construct expected full completion
        human_response = f'{{"choice": "{human_choice}"}}'
        full_input = prompt_str + human_response

        # Tokenize and compute log-likelihood of full human response
        inputs = tokenizer(full_input, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            log_likelihood = outputs.loss.item()


    return choice, log_likelihood

In [ ]:
def simulate_participant(df_participant, model, tokenizer, pipe):
    """Simulates a participant with log-likelihood tracking"""
    history = []
    cumulative_reward = 0
    total_trials = len(df_participant)


    for trial in range(total_trials):
        row = df_participant.iloc[trial]
        trial_num = row['trial']
        human_choice = row['choice']
        reward = row['reward']
        cumulative_reward += reward

        # Build prompt using actual human history
        past_trials = []
        for past_idx in range(trial):
            past_row = df_participant.iloc[past_idx]
            past_trials.append({
                "trial": past_row['trial'],
                "choice": past_row['choice'],
                "reward": past_row['reward'],
                "cumulative_reward": df_participant.iloc[:past_idx+1]['reward'].sum()
            })

        prompt = build_slot_prompt(trial_num, past_trials, total_trials)

        # Generate model choice and log-likelihood
        model_choice, log_likelihood = generate(
            prompt, pipe, model, tokenizer, human_choice
        )

        history.append({
            "trial_num": trial,
            "prompt": prompt,
            "model_choice": model_choice,
            "human_choice": human_choice,
            "reward": reward,
            "cumulative_reward": cumulative_reward,
            "log_likelihood": log_likelihood
        })
        print(f"Trial {trial}: Human {human_choice}, Model {model_choice}, LL: {log_likelihood}")

    return history

In [ ]:
def fix_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    transformers.set_seed(seed)  # For Hugging Face models

In [ ]:
participant_ids = test_data['model_id'].unique().tolist()

In [ ]:
test_data

In [ ]:
fix_seed(seeds[0])

In [ ]:
# Storage for results
all_results = []
num_participants = len(participant_ids)

fix_seed(seeds[0])

# Run simulation for each participant
for idx, participant_id in enumerate(participant_ids):
    print(f"\n🧠 Processing participant {participant_id} ({idx+1}/{num_participants})")

    # Get participant data and seed
    df_participant=test_data[test_data['model_id']==participant_id]

    # Run simulation
    history = simulate_participant(df_participant, model, tokenizer, pipe)
    all_results.append({
        "participant_id": participant_id,
        "history": history
    })

In [ ]:
results_dfs = []
for result in all_results:
    df = pd.DataFrame(result["history"])
    df["participant_id"] = result["participant_id"]
    results_dfs.append(df)

final_df = pd.concat(results_dfs, ignore_index=True)
final_df.to_csv('/content/drive/MyDrive/centaur/predictive_llama_3.1_reversal_learning.csv', index=False)